# TA-004C — DenseNet-121 Reentrenado con GAN Augmentation

**Objetivo:** Reentrenar DenseNet-121 con `train_augmented.csv` y comparar AUC contra baseline sin augmentation (AUC test = 0.8717).

**Novedades vs TA-004:** Dataset aumentado (6,206 imgs), VetXRayDataset maneja DICOM + PNG, AMP, checkpoints cada 10 épocas, W&B.

In [1]:
import json
import time
import numpy as np
import pandas as pd
import pydicom
import wandb
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from PIL import Image
from pathlib import Path
from sklearn.metrics import roc_auc_score, f1_score, accuracy_score
from tqdm import tqdm

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

BASE          = Path(r'E:\Taller Integrador I\ModelosIA\Radiografias')
CHECKPOINTS   = Path(r'E:\Taller Integrador I\ModelosIA\modelos\checkpoints')
NOTEBOOKS_DIR = Path(r'E:\Taller Integrador I\ModelosIA\modelos\Notebooks')
TRAIN_DIR     = BASE / 'dataset_split' / 'train'
VAL_DIR       = BASE / 'dataset_split' / 'val'
TEST_DIR      = BASE / 'dataset_split' / 'test'
MANIFESTS     = BASE / 'dataset_split' / 'manifests'

CLASSES                   = ['alveolar_pattern', 'bronchial_pattern', 'pleural_effusion', 'cardiomegaly', 'no_finding']
NUM_CLASSES               = len(CLASSES)
BASELINE_AUC              = 0.8717
SEED                      = 42
BATCH_SIZE                = 16
LR                        = 1e-4
WEIGHT_DECAY              = 1e-4
EPOCHS                    = 30
PATIENCE                  = 10
FEATURE_EXTRACTION_EPOCHS = 5
DROPOUT                   = 0.4
CHECKPOINT_EVERY          = 10
RESUME_FROM               = None

torch.manual_seed(SEED)
np.random.seed(SEED)
print('Config OK')

Device: cuda
GPU: NVIDIA GeForce RTX 4060
Config OK


In [2]:
wandb.init(
    project='vetxray-cnn',
    entity='dbaylont1-antenor-orrego-private-university',
    name='TA-004C-DenseNet-Augmented',
    config={
        'task': 'TA-004C', 'model': 'DenseNet-121',
        'dataset': 'VetXRay-GAN-Augmented',
        'batch_size': BATCH_SIZE, 'lr': LR,
        'weight_decay': WEIGHT_DECAY, 'epochs': EPOCHS,
        'patience': PATIENCE, 'fe_epochs': FEATURE_EXTRACTION_EPOCHS,
        'dropout': DROPOUT, 'amp': True,
        'augmentation': 'GAN', 'baseline_auc': BASELINE_AUC, 'seed': SEED
    },
    tags=['DenseNet121', 'augmented', 'TA-004C', 'VetXRay']
)
print('W&B run:', wandb.run.url)

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from C:\Users\Pcuser\_netrc.
wandb: Currently logged in as: dbaylont1 (dbaylont1-antenor-orrego-private-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


W&B run: https://wandb.ai/dbaylont1-antenor-orrego-private-university/vetxray-cnn/runs/3fefuw36


In [3]:
TRANSFORM_TRAIN = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])
TRANSFORM_VAL = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

class VetXRayDataset(Dataset):
    def __init__(self, df, transform, train_dir=TRAIN_DIR):
        self.df        = df.reset_index(drop=True)
        self.transform = transform
        self.train_dir = train_dir
        self.labels    = self._build_labels()

    def _build_labels(self):
        labels = np.zeros((len(self.df), NUM_CLASSES), dtype=np.float32)
        for i, row in self.df.iterrows():
            for j, cls in enumerate(CLASSES):
                if isinstance(row['TAG'], str) and cls in row['TAG']:
                    labels[i, j] = 1.0
        return labels

    def _load_dicom(self, path):
        dcm = pydicom.dcmread(str(path))
        arr = dcm.pixel_array.astype(np.float32)
        if hasattr(dcm, 'PhotometricInterpretation') and dcm.PhotometricInterpretation == 'MONOCHROME1':
            arr = arr.max() - arr
        p2, p98 = np.percentile(arr, 2), np.percentile(arr, 98)
        arr = np.clip(arr, p2, p98)
        arr = (arr - p2) / (p98 - p2 + 1e-8)
        img = Image.fromarray((arr * 255).astype(np.uint8))
        img = img.resize((224, 224), Image.LANCZOS)
        return Image.merge('RGB', [img, img, img])

    def _load_png(self, path):
        img = Image.open(path).convert('L')
        img = img.resize((224, 224), Image.LANCZOS)
        return Image.merge('RGB', [img, img, img])

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row      = self.df.iloc[idx]
        is_syn   = row.get('is_synthetic', False)
        syn_path = row.get('synthetic_path', None)
        if is_syn and pd.notna(syn_path):
            img_rgb = self._load_png(syn_path)
        else:
            img_rgb = self._load_dicom(self.train_dir / row['FileName'])
        return self.transform(img_rgb), torch.tensor(self.labels[idx], dtype=torch.float32)

print('VetXRayDataset OK')

VetXRayDataset OK


In [4]:
df_train = pd.read_csv(MANIFESTS / 'train_augmented.csv')
df_val   = pd.read_csv(MANIFESTS / 'val.csv')
df_test  = pd.read_csv(MANIFESTS / 'test.csv')

print(f'Train augmented: {len(df_train)}')
print(f'Val:             {len(df_val)}')
print(f'Test:            {len(df_test)}')

pos_counts    = np.array([df_train['TAG'].str.contains(c, na=False).sum() for c in CLASSES], dtype=np.float32)
neg_counts    = len(df_train) - pos_counts
class_weights = torch.tensor(neg_counts / (pos_counts + 1e-8), dtype=torch.float32).to(DEVICE)
print('\nClass weights:')
for c, w in zip(CLASSES, class_weights.cpu()): print(f'  {c}: {w:.3f}')

train_dataset = VetXRayDataset(df_train, TRANSFORM_TRAIN, train_dir=TRAIN_DIR)
val_dataset   = VetXRayDataset(df_val,   TRANSFORM_VAL,   train_dir=VAL_DIR)
test_dataset  = VetXRayDataset(df_test,  TRANSFORM_VAL,   train_dir=TEST_DIR)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)
print('DataLoaders OK')

Train augmented: 6206
Val:             1063
Test:            940

Class weights:
  alveolar_pattern: 5.206
  bronchial_pattern: 5.206
  pleural_effusion: 7.866
  cardiomegaly: 4.541
  no_finding: 1.514
DataLoaders OK


In [5]:
def build_densenet121():
    m = models.densenet121(weights='IMAGENET1K_V1')
    for p in m.parameters(): p.requires_grad = False
    in_features = m.classifier.in_features
    m.classifier = nn.Sequential(nn.Dropout(DROPOUT), nn.Linear(in_features, NUM_CLASSES))
    return m.to(DEVICE)

model  = build_densenet121()
total  = sum(p.numel() for p in model.parameters())
train_ = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Params totales:     {total:,}')
print(f'Params entrenables: {train_:,} (feature extraction)')
print(f'Batch size:         {BATCH_SIZE} (reducido por VRAM)')

Params totales:     6,958,981
Params entrenables: 5,125 (feature extraction)
Batch size:         16 (reducido por VRAM)


In [6]:
def evaluate(model, loader, criterion):
    model.eval()
    total_loss, all_probs, all_labels = 0.0, [], []
    with torch.no_grad():
        for imgs, labels in loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            with torch.amp.autocast('cuda'):
                out  = model(imgs)
                loss = criterion(out, labels)
            total_loss += loss.item() * imgs.size(0)
            all_probs.append(torch.sigmoid(out).cpu().numpy())
            all_labels.append(labels.cpu().numpy())
    probs    = np.vstack(all_probs)
    labels   = np.vstack(all_labels)
    avg_loss = total_loss / len(loader.dataset)
    try:    auc = roc_auc_score(labels, probs, average='macro')
    except: auc = 0.0
    preds = (probs >= 0.5).astype(int)
    f1  = f1_score(labels, preds, average='macro', zero_division=0)
    acc = accuracy_score(labels.flatten(), preds.flatten())
    return avg_loss, auc, f1, acc, probs, labels


def save_ckpt(epoch, model, opt, sched, scaler, best_auc, history, tag):
    p = CHECKPOINTS / f'densenet_aug_{tag}_ep{epoch}.pth'
    torch.save({'epoch': epoch, 'model_state_dict': model.state_dict(),
                'optimizer_state_dict': opt.state_dict(),
                'scheduler_state_dict': sched.state_dict(),
                'scaler_state_dict': scaler.state_dict(),
                'best_auc': best_auc, 'history': history}, p)
    return p


def load_ckpt(path, model, opt, sched, scaler):
    c = torch.load(path, map_location=DEVICE)
    model.load_state_dict(c['model_state_dict'])
    opt.load_state_dict(c['optimizer_state_dict'])
    sched.load_state_dict(c['scheduler_state_dict'])
    scaler.load_state_dict(c['scaler_state_dict'])
    print(f'Retomando desde epoch {c["epoch"]} | best_auc {c["best_auc"]:.4f}')
    return c['epoch'], c['best_auc'], c['history']


print('Helpers OK')

Helpers OK


In [7]:
def train_loop(model, optimizer, scheduler, scaler, criterion,
               start_ep, end_ep, best_auc, history, phase):
    patience_counter = 0
    for epoch in range(start_ep, end_ep):
        model.train()
        train_loss = 0.0
        t0 = time.time()
        for imgs, labels in tqdm(train_loader, desc=f'Ep {epoch+1}/{end_ep}', leave=False):
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            with torch.amp.autocast('cuda'):
                out  = model(imgs)
                loss = criterion(out, labels)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            train_loss += loss.item() * imgs.size(0)

        train_loss /= len(train_loader.dataset)
        val_loss, val_auc, val_f1, val_acc, _, _ = evaluate(model, val_loader, criterion)
        scheduler.step(val_auc)
        elapsed = (time.time() - t0) / 60
        lr = optimizer.param_groups[0]['lr']

        print(f'[{phase}] Ep {epoch+1:3d}/{end_ep} | '
              f'Train {train_loss:.4f} | Val {val_loss:.4f} | '
              f'AUC {val_auc:.4f} | F1 {val_f1:.4f} | LR {lr:.2e} | {elapsed:.1f}min')

        entry = {'epoch': epoch+1, 'phase': phase, 'train_loss': train_loss,
                 'val_loss': val_loss, 'val_auc': val_auc, 'val_f1': val_f1, 'val_acc': val_acc}
        history.append(entry)
        wandb.log({'epoch': epoch+1, 'phase': phase, 'train_loss': train_loss,
                   'val_loss': val_loss, 'val_auc': val_auc, 'val_f1': val_f1,
                   'val_acc': val_acc, 'lr': lr})

        if val_auc > best_auc:
            best_auc = val_auc
            patience_counter = 0
            save_ckpt(epoch+1, model, optimizer, scheduler, scaler, best_auc, history, 'best')
            print(f'  >>> Mejor AUC: {best_auc:.4f} — guardado')
        else:
            patience_counter += 1

        if (epoch + 1) % CHECKPOINT_EVERY == 0:
            save_ckpt(epoch+1, model, optimizer, scheduler, scaler, best_auc, history, 'periodic')

        if patience_counter >= PATIENCE:
            print(f'Early stopping en epoch {epoch+1}')
            break

    return best_auc, history


print('train_loop() OK')

train_loop() OK


In [8]:
criterion = nn.BCEWithLogitsLoss(pos_weight=class_weights)
optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()),
                               lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=3)
scaler    = torch.amp.GradScaler('cuda')

start_epoch, best_auc, history = 0, 0.0, []

if RESUME_FROM and Path(RESUME_FROM).exists():
    start_epoch, best_auc, history = load_ckpt(RESUME_FROM, model, optimizer, scheduler, scaler)

print(f'=== FASE 1: Feature Extraction ({FEATURE_EXTRACTION_EPOCHS} epocas) ===')
best_auc, history = train_loop(model, optimizer, scheduler, scaler, criterion,
                                start_epoch, FEATURE_EXTRACTION_EPOCHS,
                                best_auc, history, 'feature_extraction')

=== FASE 1: Feature Extraction (5 epocas) ===


[feature_extraction] Ep   1/5 | Train 1.0580 | Val 0.9639 | AUC 0.5515 | F1 0.2556 | LR 1.00e-04 | 13.0min
  >>> Mejor AUC: 0.5515 — guardado


[feature_extraction] Ep   2/5 | Train 0.9940 | Val 0.9450 | AUC 0.5918 | F1 0.2670 | LR 1.00e-04 | 12.3min
  >>> Mejor AUC: 0.5918 — guardado


[feature_extraction] Ep   3/5 | Train 0.9644 | Val 0.9376 | AUC 0.6191 | F1 0.2567 | LR 1.00e-04 | 12.3min
  >>> Mejor AUC: 0.6191 — guardado


[feature_extraction] Ep   4/5 | Train 0.9479 | Val 0.9260 | AUC 0.6403 | F1 0.2821 | LR 1.00e-04 | 12.2min
  >>> Mejor AUC: 0.6403 — guardado


[feature_extraction] Ep   5/5 | Train 0.9284 | Val 0.9171 | AUC 0.6561 | F1 0.3048 | LR 1.00e-04 | 12.2min
  >>> Mejor AUC: 0.6561 — guardado


In [9]:
print('=== FASE 2: Fine-tuning completo ===')
for p in model.parameters(): p.requires_grad = True
optimizer = torch.optim.AdamW(model.parameters(), lr=LR/5, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=3)
print(f'Params entrenables: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}')

best_auc, history = train_loop(model, optimizer, scheduler, scaler, criterion,
                                FEATURE_EXTRACTION_EPOCHS, EPOCHS,
                                best_auc, history, 'fine_tuning')
print(f'Entrenamiento completo. Mejor AUC val: {best_auc:.4f}')

=== FASE 2: Fine-tuning completo ===
Params entrenables: 6,958,981


[fine_tuning] Ep   6/30 | Train 0.8065 | Val 0.7920 | AUC 0.7961 | F1 0.4133 | LR 2.00e-05 | 12.5min
  >>> Mejor AUC: 0.7961 — guardado


[fine_tuning] Ep   7/30 | Train 0.6501 | Val 0.6943 | AUC 0.8365 | F1 0.5091 | LR 2.00e-05 | 12.4min
  >>> Mejor AUC: 0.8365 — guardado


[fine_tuning] Ep   8/30 | Train 0.5731 | Val 0.6568 | AUC 0.8534 | F1 0.5318 | LR 2.00e-05 | 12.4min
  >>> Mejor AUC: 0.8534 — guardado


[fine_tuning] Ep   9/30 | Train 0.5149 | Val 0.6233 | AUC 0.8654 | F1 0.5706 | LR 2.00e-05 | 12.3min
  >>> Mejor AUC: 0.8654 — guardado


...

[fine_tuning] Ep  21/30 | Train 0.2367 | Val 0.6727 | AUC 0.8769 | F1 0.6171 | LR 1.00e-05 | 12.3min


[fine_tuning] Ep  22/30 | Train 0.2188 | Val 0.7060 | AUC 0.8760 | F1 0.6070 | LR 1.00e-05 | 12.6min


[fine_tuning] Ep  23/30 | Train 0.2102 | Val 0.7173 | AUC 0.8766 | F1 0.6107 | LR 5.00e-06 | 12.4min


[fine_tuning] Ep  24/30 | Train 0.2032 | Val 0.7154 | AUC 0.8767 | F1 0.6152 | LR 5.00e-06 | 12.4min


[fine_tuning] Ep  25/30 | Train 0.1888 | Val 0.7419 | AUC 0.8768 | F1 0.6085 | LR 5.00e-06 | 13.2min
Early stopping en epoch 25
Entrenamiento completo. Mejor AUC val: 0.8828


In [10]:
best_files = sorted(CHECKPOINTS.glob('densenet_aug_best*.pth'),
                    key=lambda p: int(p.stem.split('ep')[-1]))
if best_files:
    ckpt = torch.load(best_files[-1], map_location=DEVICE)
    model.load_state_dict(ckpt['model_state_dict'])
    print(f'Mejor modelo cargado: {best_files[-1].name}')

t0 = time.time()
test_loss, test_auc, test_f1, test_acc, test_probs, test_labels = evaluate(model, test_loader, criterion)
inf_ms = (time.time() - t0) / len(test_dataset) * 1000

print(f'\n=== RESULTADOS TEST ===')
print(f'AUC macro:  {test_auc:.4f}  (baseline: {BASELINE_AUC})')
print(f'F1 macro:   {test_f1:.4f}')
print(f'Accuracy:   {test_acc:.4f}')
print(f'Inferencia: {inf_ms:.2f} ms/imagen')

try:
    aucs = roc_auc_score(test_labels, test_probs, average=None)
    print('\nAUC por clase:')
    for c, a in zip(CLASSES, aucs): print(f'  {c}: {a:.4f}')
except Exception as e:
    print(e)

wandb.log({'test_auc': test_auc, 'test_f1': test_f1, 'test_accuracy': test_acc,
           'inference_ms': inf_ms, 'mejora_vs_baseline': test_auc - BASELINE_AUC})

Mejor modelo cargado: densenet_aug_best_ep15.pth

=== RESULTADOS TEST ===
AUC macro:  0.8902  (baseline: 0.8717)
F1 macro:   0.6196
Accuracy:   0.8594
Inferencia: 118.78 ms/imagen

AUC por clase:
  alveolar_pattern: 0.8916
  bronchial_pattern: 0.7704
  pleural_effusion: 0.9877
  cardiomegaly: 0.9056
  no_finding: 0.8958


In [11]:
results = {
    'model': 'densenet121', 'task': 'TA-004C',
    'dataset': 'VetXRay-GAN-Augmented',
    'baseline_auc': BASELINE_AUC,
    'val_auc': best_auc,
    'test_auc': test_auc,
    'test_f1_macro': test_f1,
    'test_accuracy': test_acc,
    'inference_ms': inf_ms,
    'mejora_auc': round(test_auc - BASELINE_AUC, 4),
    'history': history
}
out = NOTEBOOKS_DIR / 'densenet_aug_results.json'
with open(out, 'w') as f: json.dump(results, f, indent=4)

mejora = test_auc - BASELINE_AUC
print(f'Resultados: {out}')
print(f'AUC test: {test_auc:.4f} vs baseline {BASELINE_AUC} => {"MEJORA" if mejora > 0 else "SIN MEJORA"} de {abs(mejora):.4f}')

wandb.finish()
print('TA-004C completado.')

Resultados: E:\Taller Integrador I\ModelosIA\modelos\Notebooks\densenet_aug_results.json
AUC test: 0.8902 vs baseline 0.8717 => MEJORA de 0.0185


epoch,▁▁▂▂▂▂▃▃▃▄▄▄▅▅▅▅▆▆▆▇▇▇▇██
inference_ms,▁
lr,█████▂▂▂▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁
mejora_vs_baseline,▁
test_accuracy,▁
test_auc,▁
test_f1,▁
train_loss,█▇▇▇▇▆▅▄▄▃▃▃▃▃▂▂▂▂▂▁▁▁▁▁▁
val_acc,▁▂▂▃▃▃▅▆▇▇▇▇▇▇█▇█████████
val_auc,▁▂▂▃▃▆▇▇█████████████████
+2,...


TA-004C completado.


In [12]:
resnet_base   = {'model': 'ResNet-50',       'auc_base': 0.8619, 'f1_base': 0.5977}
effnet_base   = {'model': 'EfficientNet-B0', 'auc_base': 0.8621, 'f1_base': 0.5742}
dense_base    = {'model': 'DenseNet-121',    'auc_base': 0.8717, 'f1_base': 0.5846}

resnet_aug_r  = json.load(open(NOTEBOOKS_DIR / 'resnet50_aug_results.json'))
effnet_aug_r  = json.load(open(NOTEBOOKS_DIR / 'efficientnet_aug_results.json'))
dense_aug_r   = json.load(open(NOTEBOOKS_DIR / 'densenet_aug_results.json'))

print('=== TABLA COMPARATIVA: Baseline vs GAN Augmentation ===')
print(f'{"Modelo":<20} {"AUC base":>9} {"AUC aug":>9} {"Δ AUC":>7} {"F1 base":>8} {"F1 aug":>8}')
print('-' * 65)
for base, aug in [(resnet_base, resnet_aug_r), (effnet_base, effnet_aug_r), (dense_base, dense_aug_r)]:
    delta = aug['test_auc'] - base['auc_base']
    print(f'{base["model"]:<20} {base["auc_base"]:>9.4f} {aug["test_auc"]:>9.4f} {delta:>+7.4f} '
          f'{base["f1_base"]:>8.4f} {aug["test_f1_macro"]:>8.4f}')

best_aug = max([resnet_aug_r, effnet_aug_r, dense_aug_r], key=lambda x: x['test_auc'])
print(f'\nModelo ganador augmentado: {best_aug["model"]} | AUC test: {best_aug["test_auc"]:.4f}')

=== TABLA COMPARATIVA: Baseline vs GAN Augmentation ===
Modelo                AUC base   AUC aug   Δ AUC  F1 base   F1 aug
-----------------------------------------------------------------
ResNet-50               0.8619    0.8831 +0.0212   0.5977   0.5797
EfficientNet-B0         0.8621    0.8862 +0.0241   0.5742   0.6293
DenseNet-121            0.8717    0.8902 +0.0185   0.5846   0.6196

Modelo ganador augmentado: densenet121 | AUC test: 0.8902
